# SDH exp_011 — B04 고정 독립 FE ablation

정확한 GS B04 챔피언과 LR을 기준으로 고정하고 신규 표현을 하나씩 독립적으로 추가합니다. 각 셀을 위에서부터 실행하세요.

In [ ]:
from pathlib import Path
from time import perf_counter
import gc
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'common').is_dir() and (p / 'experiments').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('프로젝트 루트를 찾지 못했습니다.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiments.SDH.exp_011_b04_independent_fe.preprocessing import (
    B04,
    B04_ID,
    FeatureCase,
    build_case_matrices,
    combine_cases,
    make_context,
    make_independent_cases,
)

TRAIN_PATH = PROJECT_ROOT / 'data' / 'raw' / 'train.csv'
RESULTS_DIR = PROJECT_ROOT / 'experiments' / 'SDH' / 'exp_011_b04_independent_fe' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
N_SPLITS = 5
SCREEN_SEEDS = (42,)
CONFIRMATION_SEEDS = (42, 52, 62)
B04_EXPECTED_SEED42 = 0.47814168846488037
print(B04_ID, B04.CONFIG.lr_c, B04.CONFIG.lr_max_iter)

## 1. 데이터와 row-local cache

cache는 각 행의 문자열만 파싱합니다. label 통계와 fold별 피처 선택은 아래 CV 안에서 별도로 수행합니다.

In [ ]:
train = pd.read_csv(TRAIN_PATH, low_memory=False)
genes = [column for column in train.columns if column not in ('ID', 'SUBCLASS')]
labels = train['SUBCLASS'].reset_index(drop=True)
context = make_context(train[genes], genes, show_progress=True)
classes = sorted(labels.unique())
print('train:', train.shape, 'classes:', len(classes))
print('gene-type vocabulary:', context.gene_type_matrix.shape[1])
print('exact-event vocabulary:', context.cache.event_matrix.shape[1])

In [ ]:
cases = make_independent_cases()
display(pd.DataFrame([
    {'case': case.name, 'blocks': ', '.join(case.blocks) or 'B04 only', 'description': case.description}
    for case in cases.values()
]))

## 2. 공통 평가 함수

모델은 B04의 LR 생성 함수를 그대로 사용합니다. supervised enrichment는 `build_case_matrices` 안에서 outer-train 내부 5-fold OOF 표현으로 만들어집니다.

In [ ]:
def evaluate_case(case: FeatureCase, seeds=SCREEN_SEEDS):
    per_seed_rows = []
    class_rows = []
    started = perf_counter()

    for seed in seeds:
        splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        prediction = np.empty(len(labels), dtype=object)
        fold_scores = []
        feature_counts = []
        extra_counts = []
        warning_count = 0

        for fold, (train_index, valid_index) in enumerate(
            splitter.split(np.zeros(len(labels)), labels), start=1
        ):
            fold_started = perf_counter()
            x_train, x_valid, feature_names, metadata = build_case_matrices(
                context,
                train_index,
                valid_index,
                labels,
                case,
                inner_seed=seed * 100 + fold,
            )
            model = B04.make_model('logistic', seed, B04.CONFIG.lr_max_iter)
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always', ConvergenceWarning)
                model.fit(x_train, labels.iloc[train_index])
            fold_prediction = model.predict(x_valid)
            prediction[valid_index] = fold_prediction
            fold_f1 = f1_score(
                labels.iloc[valid_index], fold_prediction, average='macro', zero_division=0
            )
            fold_scores.append(fold_f1)
            feature_counts.append(len(feature_names))
            extra_counts.append(metadata['extra_feature_count'])
            warning_count += sum(
                issubclass(item.category, ConvergenceWarning) for item in caught
            )
            print(
                f'[{case.name}] seed={seed} fold={fold}/{N_SPLITS} '
                f'f1={fold_f1:.5f} features={len(feature_names):,} '
                f'extra={metadata["extra_feature_count"]} '
                f'time={perf_counter() - fold_started:.1f}s'
            )
            del x_train, x_valid, model
            gc.collect()

        oof_f1 = f1_score(labels, prediction, average='macro', zero_division=0)
        accuracy = accuracy_score(labels, prediction)
        per_class = f1_score(
            labels, prediction, labels=classes, average=None, zero_division=0
        )
        support = labels.value_counts().reindex(classes).to_numpy()
        per_seed_rows.append({
            'case': case.name,
            'seed': seed,
            'oof_f1_macro': oof_f1,
            'oof_accuracy': accuracy,
            'fold_f1_mean': float(np.mean(fold_scores)),
            'fold_f1_std': float(np.std(fold_scores)),
            'feature_count_min': min(feature_counts),
            'feature_count_max': max(feature_counts),
            'extra_feature_count_min': min(extra_counts),
            'extra_feature_count_max': max(extra_counts),
            'convergence_warning_count': warning_count,
        })
        class_rows.extend({
            'case': case.name, 'seed': seed, 'class': class_name,
            'f1': score, 'support': count
        } for class_name, score, count in zip(classes, per_class, support))
        print(f'완료: {case.name} seed={seed} OOF Macro F1={oof_f1:.5f}')

    per_seed = pd.DataFrame(per_seed_rows)
    summary = {
        'case': case.name,
        'blocks': ','.join(case.blocks) or 'B04',
        'seeds': list(seeds),
        'oof_f1_macro_mean': per_seed['oof_f1_macro'].mean(),
        'oof_f1_macro_std': per_seed['oof_f1_macro'].std(ddof=0),
        'oof_accuracy_mean': per_seed['oof_accuracy'].mean(),
        'feature_count_min': per_seed['feature_count_min'].min(),
        'feature_count_max': per_seed['feature_count_max'].max(),
        'extra_feature_count_min': per_seed['extra_feature_count_min'].min(),
        'extra_feature_count_max': per_seed['extra_feature_count_max'].max(),
        'convergence_warning_count': per_seed['convergence_warning_count'].sum(),
        'elapsed_seconds': perf_counter() - started,
    }
    return {
        'case': case,
        'summary': summary,
        'per_seed': per_seed,
        'class_f1': pd.DataFrame(class_rows),
    }

## 3. seed 42 독립 ablation

아래 여섯 셀은 서로 누적되지 않습니다. 먼저 case 00으로 B04 재현을 확인합니다.

In [ ]:
screen_results = {}
case_name = 'case_00_b04'
screen_results[case_name] = evaluate_case(cases[case_name])
observed_b04 = screen_results[case_name]['summary']['oof_f1_macro_mean']
reproduction_delta = observed_b04 - B04_EXPECTED_SEED42
print(
    'stored B04:', B04_EXPECTED_SEED42,
    'observed:', observed_b04,
    'delta:', reproduction_delta,
)
if abs(reproduction_delta) > 0.002:
    raise RuntimeError(
        'B04 차이가 0.002를 넘습니다. 피처 수와 실행 환경을 확인하세요.'
    )
print('B04 구조 및 점수 재현 허용 범위 통과')

In [ ]:
case_name = 'case_01_b04_plus_burden_bins'
screen_results[case_name] = evaluate_case(cases[case_name])

In [ ]:
case_name = 'case_02_b04_plus_row_profile'
screen_results[case_name] = evaluate_case(cases[case_name])

In [ ]:
case_name = 'case_03_b04_plus_gene_enrichment'
screen_results[case_name] = evaluate_case(cases[case_name])

In [ ]:
case_name = 'case_04_b04_plus_gene_type_enrichment'
screen_results[case_name] = evaluate_case(cases[case_name])

In [ ]:
case_name = 'case_05_b04_plus_exact_event_enrichment'
screen_results[case_name] = evaluate_case(cases[case_name])

In [ ]:
leaderboard = (
    pd.DataFrame([result['summary'] for result in screen_results.values()])
    .sort_values('oof_f1_macro_mean', ascending=False)
    .reset_index(drop=True)
)
b04_score = leaderboard.loc[
    leaderboard['case'].eq('case_00_b04'), 'oof_f1_macro_mean'
].iloc[0]
leaderboard['delta_vs_b04'] = leaderboard['oof_f1_macro_mean'] - b04_score
leaderboard.to_csv(RESULTS_DIR / 'leaderboard_seed42.csv', index=False)
pd.concat([result['class_f1'] for result in screen_results.values()]).to_csv(
    RESULTS_DIR / 'class_f1_seed42.csv', index=False
)
display(leaderboard)

## 4. 양의 개선 상위 2개 조합

B04보다 높은 독립 블록이 2개 이상일 때만 상위 2개를 조합합니다. 음수 후보는 자동으로 제외합니다.

In [ ]:
positive_names = leaderboard.loc[
    (leaderboard['case'] != 'case_00_b04') & (leaderboard['delta_vs_b04'] > 0),
    'case',
].head(2).tolist()
combination_result = None
if len(positive_names) >= 2:
    combination_case = combine_cases(
        'case_06_b04_plus_top2_combination',
        [cases[name] for name in positive_names],
    )
    print('조합 블록:', combination_case.blocks)
    combination_result = evaluate_case(combination_case)
    combination_result['summary']['delta_vs_b04'] = (
        combination_result['summary']['oof_f1_macro_mean'] - b04_score
    )
    display(pd.DataFrame([combination_result['summary']]))
else:
    print('양의 독립 후보가 2개 미만이므로 조합을 건너뜁니다:', positive_names)

## 5. B04와 상위 후보 3-seed 확인

B04를 반드시 포함하고, seed 42 전체 후보와 선택적 조합 중 상위 비기준 2개를 seed 42/52/62로 다시 실행합니다.

In [ ]:
ranked_cases = {name: result['case'] for name, result in screen_results.items()}
ranked_rows = [result['summary'] for result in screen_results.values()]
if combination_result is not None:
    ranked_cases[combination_result['case'].name] = combination_result['case']
    ranked_rows.append(combination_result['summary'])
ranked = pd.DataFrame(ranked_rows).sort_values('oof_f1_macro_mean', ascending=False)
top_nonbaseline = ranked.loc[ranked['case'] != 'case_00_b04', 'case'].head(2).tolist()
confirmation_names = ['case_00_b04', *top_nonbaseline]
print('3-seed confirmation:', confirmation_names)
confirmation_results = {
    name: evaluate_case(ranked_cases[name], seeds=CONFIRMATION_SEEDS)
    for name in confirmation_names
}

In [ ]:
confirmation_leaderboard = (
    pd.DataFrame([result['summary'] for result in confirmation_results.values()])
    .sort_values('oof_f1_macro_mean', ascending=False)
    .reset_index(drop=True)
)
confirmation_b04 = confirmation_leaderboard.loc[
    confirmation_leaderboard['case'].eq('case_00_b04'), 'oof_f1_macro_mean'
].iloc[0]
confirmation_leaderboard['delta_vs_b04'] = (
    confirmation_leaderboard['oof_f1_macro_mean'] - confirmation_b04
)
confirmation_leaderboard.to_csv(
    RESULTS_DIR / 'leaderboard_confirmation.csv', index=False
)
pd.concat([result['per_seed'] for result in confirmation_results.values()]).to_csv(
    RESULTS_DIR / 'per_seed_confirmation.csv', index=False
)
pd.concat([result['class_f1'] for result in confirmation_results.values()]).to_csv(
    RESULTS_DIR / 'class_f1_confirmation.csv', index=False
)
display(confirmation_leaderboard)

## 6. 승자 단독 제출 파일 생성

`B04 + gene×event-type enrichment`만 사용합니다. 전체 train 입력은 내부 5-fold OOF enrichment로 만들고, test에는 전체 train에서 학습한 enrichment 가중치를 적용만 합니다.

In [ ]:
TEST_PATH = PROJECT_ROOT / 'data' / 'raw' / 'test.csv'
SAMPLE_PATH = PROJECT_ROOT / 'data' / 'raw' / 'sample_submission.csv'
SUBMISSION_PATH = (
    RESULTS_DIR / 'submission_exp011_b04_gene_type_enrichment_seed42.csv'
)

test = pd.read_csv(TEST_PATH, low_memory=False)
sample_submission = pd.read_csv(SAMPLE_PATH)
assert list(test.columns) == ['ID', *genes]
assert list(sample_submission.columns) == ['ID', 'SUBCLASS']
assert sample_submission['ID'].equals(test['ID'])
assert int(train[genes].isna().sum().sum()) == 0
assert int(test[genes].isna().sum().sum()) == B04.CONFIG.expected_test_nan

combined_features = pd.concat(
    [train[genes], test[genes]], axis=0, ignore_index=True
)
submission_context = make_context(
    combined_features, genes, show_progress=True
)
full_train_index = np.arange(len(train))
test_index = np.arange(len(train), len(combined_features))
winner_case = cases['case_04_b04_plus_gene_type_enrichment']

x_full_train, x_test, submission_features, submission_metadata = (
    build_case_matrices(
        submission_context,
        full_train_index,
        test_index,
        labels,
        winner_case,
        inner_seed=42,
    )
)
submission_model = B04.make_model(
    'logistic', 42, B04.CONFIG.lr_max_iter
)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always', ConvergenceWarning)
    submission_model.fit(x_full_train, labels)
submission_prediction = submission_model.predict(x_test)
submission_warning_count = sum(
    issubclass(item.category, ConvergenceWarning) for item in caught
)

submission = sample_submission.copy()
submission['SUBCLASS'] = submission_prediction
assert len(submission) == len(test)
assert submission['ID'].equals(test['ID'])
assert int(submission.isna().sum().sum()) == 0
submission.to_csv(SUBMISSION_PATH, index=False)

display(pd.DataFrame([{
    'case': winner_case.name,
    'seed': 42,
    'train_rows': len(train),
    'test_rows': len(test),
    'feature_count': len(submission_features),
    'extra_feature_count': submission_metadata['extra_feature_count'],
    'convergence_warning_count': submission_warning_count,
    'submission_path': str(SUBMISSION_PATH),
}]))
print('저장 완료:', SUBMISSION_PATH)